# APT — one-shot Colab notebook (3 cells)

1. **Cell 1** — mount Drive, clone the repo.
2. **Cell 2** — run the orchestrator `scripts/colab_run_all.py` which does
   env setup, optional dataset copy to /content, smoke tests, Stage 1
   pre-training, Stage 2 fine-tuning, evaluation, and writes a
   **single training report** to Drive.
3. **Cell 3** — print the training report so you can paste it back into chat.

**Prerequisites** (on Google Drive, only once):
```
/content/drive/MyDrive/action_genome/
    annotations/                            # AG annotation pickles
    frames/<video_id>/*.png                 # extracted frames
/content/drive/MyDrive/apt_assets/
    faster_rcnn_ag.pth                      # STTran Google-Drive ckpt
```
Enable **Runtime → Change runtime type → GPU (A100)** before running.

## Cell 1 — mount Drive + clone repo

In [ ]:
from google.colab import drive
import os, subprocess
drive.mount('/content/drive', force_remount=False)

REPO_URL  = 'https://github.com/TommasoAiello08/STTran.git'   # change if needed
REPO_ROOT = '/content/STTran'

if not os.path.isdir(REPO_ROOT):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT])
os.chdir(REPO_ROOT)
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode())

# Stage the Faster R-CNN checkpoint into fasterRCNN/models/ if the user
# placed it on Drive (recommended: /content/drive/MyDrive/apt_assets/).
ASSETS = '/content/drive/MyDrive/apt_assets/faster_rcnn_ag.pth'
if os.path.isfile(ASSETS):
    os.makedirs('fasterRCNN/models', exist_ok=True)
    import shutil
    shutil.copy2(ASSETS, 'fasterRCNN/models/faster_rcnn_ag.pth')
    print('Copied Faster R-CNN checkpoint into fasterRCNN/models/.')
else:
    print('WARNING: faster_rcnn_ag.pth not found at', ASSETS)
    print('         Place it there before running Cell 2, or set')
    print('         FASTER_RCNN_GDRIVE_ID / FASTER_RCNN_URL in Cell 2.')

## Cell 2 — run the whole pipeline

The orchestrator does **everything**: env setup, dataset copy to local SSD,
smoke tests, Stage 1 pre-training, Stage 2 fine-tuning, evaluation, and
writes a training report to Drive at
`/content/drive/MyDrive/apt_ckpts/training_report.txt`.

Re-run this cell after a session disconnection: the `--resume` flag picks
training up from the latest checkpoint on Drive.

In [ ]:
%env AG_ROOT=/content/drive/MyDrive/action_genome
%env CKPT_ROOT=/content/drive/MyDrive/apt_ckpts

# OPTIONAL: if you DID NOT manually place faster_rcnn_ag.pth in Cell 1,
# set the Google Drive file id (or direct URL) below:
# %env FASTER_RCNN_GDRIVE_ID=1_ReplaceWithRealFileId

# NOTE on --copy_to_local:
#   Colab A100 instances have ~200 GB local SSD. If your frames/ directory
#   fits (mini / subset setups), add --copy_to_local for 10x faster I/O.
#   Skip it for the full ~300 GB AG extraction.
!python scripts/colab_run_all.py --stage all \
    --ag_root $AG_ROOT \
    --ckpt_root $CKPT_ROOT \
    --mode predcls \
    --resume

## Cell 3 — print the training report

In [ ]:
import os
report_path = '/content/drive/MyDrive/apt_ckpts/training_report.txt'
if os.path.isfile(report_path):
    with open(report_path) as fh:
        print(fh.read())
else:
    print('No report yet at', report_path)